# Study 957 — Holdco Discount — the teardown

Seven listed holding companies whose value is dominated by one **listed** stake, so NAV is markable from the tape. Two questions in order: (a) does the observable-NAV discount mean-revert, (b) does a hedged buy-the-wide-discount rule pay after costs and borrow. Inside: the NAV construction and its assumptions, the Driscoll-Kraay pooled predictive regression, the dollar-neutral pair race against an always-on control, bootstrap CIs, an era cut, cost and borrow sweeps, a NAV-calibration sweep, leave-one-out, the stale-quote decomposition that decides the verdict, and the live synthetic control.

Every real number is frozen from `docs/results.md` (Fingerprint `d51337b2a3bb`).

In [1]:
R = {'start': '2004-01-02', 'end': '2026-06-30', 'n_names': 7, 'fp': 'd51337b2a3bb', 'pooled_beta': -0.0002, 'pooled_t': -0.09, 'pooled_n': 24572, 'pooled_days': 5581, 'pooled_r2': 0.0, 'n_neg': 5, 'prosus_beta': -0.0358, 'prosus_t': -4.07, 'prosus_n': 1219, 'dior_beta': 0.0037, 'lbrd_beta': -0.0031, 'bol_beta': 0.0031, 'bol_hl': 1514, 'heio_hl': 95, 'timed_sharpe': 0.312, 'timed_cagr': 2.01, 'timed_vol': 7.22, 'timed_dd': -17.9, 'timed_t': 1.69, 'timed_gross': 0.566, 'timed_gross_t': 3.04, 'timed_inv': 80.7, 'always_sharpe': -0.268, 'always_cagr': -1.68, 'always_dd': -43.8, 'always_t': -1.41, 'always_gross': 0.046, 'always_gross_t': 0.24, 'adv': 0.58, 't_diff': 3.25, 'ci_net_lo': -0.05, 'ci_net_hi': 0.656, 'ci_net_neg': 4.7, 'ci_gross_lo': 0.204, 'ci_gross_hi': 0.91, 'ci_always_lo': -0.645, 'ci_always_hi': 0.093, 'ci_always_neg': 93.0, 'era_e_sh': 0.319, 'era_e_t': 1.33, 'era_l_sh': 0.256, 'era_l_t': 0.86, 'cost0': 0.51, 'cost0_t': 2.75, 'cost5': 0.411, 'cost5_t': 2.22, 'cost25': 0.016, 'cost25_t': 0.08, 'cost50': -0.467, 'cost50_t': -2.52, 'borrow0': 0.367, 'borrow0_t': 1.99, 'borrow600': 0.036, 'borrow600_t': 0.2, 'clean_net': 0.148, 'clean_net_t': 0.76, 'clean_gross': 0.368, 'clean_gross_t': 1.87, 'clean_mr_beta': 0.0033, 'clean_mr_t': 1.27, 'adr_net': 0.389, 'adr_net_t': 1.85, 'adr_gross': 0.458, 'adr_gross_t': 2.16, 'pn_prosus': 0.661, 'pn_prosus_t': 1.61, 'pn_softbank': 0.398, 'pn_softbank_t': 1.17, 'pn_heio': 0.145, 'pn_dior': 0.087, 'pn_lbrd': 0.066, 'pn_bol': 0.093, 'loo_lo': 0.259, 'loo_hi': 0.346, 'loo_t_lo': 1.41, 'loo_t_hi': 1.87, 'calib_lo': 0.275, 'calib_hi': 0.321, 'lt_net': -0.04, 'lt_net_t': -0.24, 'lt_gross': 0.174, 'lo_timed': 0.536, 'lo_holdcos': 0.655, 'lo_stakes': 0.636, 'lo_t_diff': 0.02, 'thr_net_lo': 0.08, 'thr_net_hi': 1.9, 'thr_gross_lo': 1.56, 'thr_gross_hi': 3.43, 'thr_cells': 17, 'thr_cells_net2': 0, 'thr_best_gross': 0.653, 'thr_best_gross_t': 3.43, 'zw252_t': 2.42, 'zw378_t': 2.46, 'zw756_t': 1.89, 'zw1008_t': 0.94, 'zw756': 0.333, 'zw1008': 0.16, 'lbrd_keep_beta': 0.0005, 'lbrd_keep_t': 0.19, 'lbrd_keep_neg': 4, 'lbrd_cut_days': 406, 'irx_cagr': 1.45, 'bil_cagr': 1.36, 'syn_pl_t': -11.76, 'syn_pl_gross': 0.999, 'syn_nl_t': -0.38, 'syn_nl_gross': 0.129}

## Construction, and what is an assumption

`discount_t = 1 - P_holdco_t / (k * P_stake_t + o)` from **price-only** (split-adjusted, dividend-unadjusted) closes — putting total-return closes on both legs of a *valuation* ratio would inject the two names' dividend-yield difference straight into the "discount". Every **return** in the backtest comes from total-return closes.

`k` and `o` are **assumptions**, frozen at a 2026 snapshot, and in reality both drift. Three are published share counts (Heineken Holding, Christian Dior, Liberty Broadband); four are *anchored* to a widely reported discount on one date, because ADR ratios and chained holdings make a bottom-up count unverifiable from the tape. The consequence is stated rather than hidden: the **level** of each series is only as good as that table, so every test runs on the **trailing z** — each name standardised against its own preceding 504 days, using only data through *t*. A constant error in `k` or `o` barely moves a z-score, and the calibration sweep below shows it does not. For the four names whose `o` is zero the cancellation is *exact* — scale `k` and the z-score is literally unchanged — so anchoring `k` to a reported discount on one date cannot leak hindsight into any test.

Three names stop early, all on **corporate events announced in advance**, never on performance, and all three shorten the sample: Bollore at the December 2024 Vivendi four-way split, SoftBank at the 2022 Alibaba disposal, and Liberty Broadband at the 2024-11-13 Charter merger agreement — after which its gap is a **merger spread** with a contractual exchange ratio and a convergence date, which is Study 366's subject and not a holdco discount. That last cut removes the one window in the panel where convergence was *guaranteed*, so it is the conservative choice; keeping it leaves the pair identical to three decimals and moves the pooled slope to +0.0005 (*t* = +0.19).

> 💡 **In plain words** — we cannot pin down exactly how big each discount is, so we never rely on that. We only ever ask whether *this* holdco is unusually cheap compared with *its own* recent history.

## (a) The predictive regression

`d_{t+126} - d_t = a + b * z_t`, pooled across names. Two problems would wreck a naive *t*: the forward windows overlap (residuals autocorrelated out to 126 lags), and the names are contemporaneously correlated — two of them literally sit on the same underlying, Tencent. So the score contributions are summed **across names within each calendar day** and only then Bartlett-weighted: Driscoll-Kraay, not a concatenated Newey-West that would splice one name's last day onto another's first.

In [2]:
print('pooled beta %+.4f   DK t %+.2f   R2 %.4f   n=%d over %d days'
      % (R['pooled_beta'], R['pooled_t'], R['pooled_r2'], R['pooled_n'], R['pooled_days']))
print('right sign (beta < 0) in %d/7 names' % R['n_neg'])
print()
print('the only individually significant name:')
print('  Prosus  beta %+.4f  t %+.2f  n=%d  <- the SHORTEST sample in the panel'
      % (R['prosus_beta'], R['prosus_t'], R['prosus_n']))
print('the two with the WRONG sign:')
print('  Christian Dior %+.4f   Bollore %+.4f'
      % (R['dior_beta'], R['bol_beta']))
print()
print('horizon is a convention, not a result -- pooled DK t by forward window:')
print('  h= 21d  t -1.18    h= 63d  t -0.40    h=126d  t %+.2f    h=252d  t +0.30'
      % R['pooled_t'])

pooled beta -0.0002   DK t -0.09   R2 0.0000   n=24572 over 5581 days
right sign (beta < 0) in 5/7 names

the only individually significant name:
  Prosus  beta -0.0358  t -4.07  n=1219  <- the SHORTEST sample in the panel
the two with the WRONG sign:
  Christian Dior +0.0037   Bollore +0.0031

horizon is a convention, not a result -- pooled DK t by forward window:
  h= 21d  t -1.18    h= 63d  t -0.40    h=126d  t -0.09    h=252d  t +0.30


Pooled *t* = **-0.09** and R² = 0.0000. Prosus's *t* = -4.07 rests on 1,219 usable days, where a 504-day trailing window and a 126-day forward window overlap so heavily that a handful of independent episodes carry the whole regression — the Valkanov problem in miniature. Half-lives corroborate: Bollore 1,514 trading days against Heineken Holding's 95, no common scale at all.

## (b) The pair race

The hedge ratio is not a free parameter. Since `P_holdco = (1 - d) * k * P_stake`, taking logs gives `log P_holdco = log(1 - d) + log(k * P_stake)` — so a **dollar-neutral** long/short earns exactly `Δ log(1 - d)` and nothing else. (Shorting the whole look-through stake value `k*P_stake/P_holdco` is the correct hedge only for a gap fixed in *money*; on a proportional gap it over-hedges and leaves a residual short. Reported as a variant further down.)

Both arms are long/short pairs funded from collateral that itself earns cash, so **both are excess-of-cash by construction** — the race is like for like without subtracting anything. Positions are gross-normalised to one unit of notional, equal-weighted across active names, and lagged exactly one day.

In [3]:
print('timed      net Sharpe %+.3f (HAC t %+.2f)  CAGR %+.2f%%  vol %.2f%%  maxDD %.1f%%'
      % (R['timed_sharpe'], R['timed_t'], R['timed_cagr'], R['timed_vol'], R['timed_dd']))
print('           gross Sharpe %+.3f (t %+.2f)   invested %.1f%% of days'
      % (R['timed_gross'], R['timed_gross_t'], R['timed_inv']))
print('always-on  net Sharpe %+.3f (HAC t %+.2f)  CAGR %+.2f%%  maxDD %.1f%%'
      % (R['always_sharpe'], R['always_t'], R['always_cagr'], R['always_dd']))
print('           gross Sharpe %+.3f (t %+.2f)'
      % (R['always_gross'], R['always_gross_t']))
print()
print('timing advantage %+.3f   HAC t on the daily difference %+.2f'
      % (R['adv'], R['t_diff']))
print()
print('bootstrap Sharpe CIs (2,000 draws, 21-day blocks):')
print('  timed net    %+.3f  [%+.3f, %+.3f]  share<0 %.1f%%   <- includes zero'
      % (R['timed_sharpe'], R['ci_net_lo'], R['ci_net_hi'], R['ci_net_neg']))
print('  timed gross  %+.3f  [%+.3f, %+.3f]'
      % (R['timed_gross'], R['ci_gross_lo'], R['ci_gross_hi']))
print('  always-on    %+.3f  [%+.3f, %+.3f]  share<0 %.1f%%'
      % (R['always_sharpe'], R['ci_always_lo'], R['ci_always_hi'], R['ci_always_neg']))

timed      net Sharpe +0.312 (HAC t +1.69)  CAGR +2.01%  vol 7.22%  maxDD -17.9%
           gross Sharpe +0.566 (t +3.04)   invested 80.7% of days
always-on  net Sharpe -0.268 (HAC t -1.41)  CAGR -1.68%  maxDD -43.8%
           gross Sharpe +0.046 (t +0.24)

timing advantage +0.580   HAC t on the daily difference +3.25

bootstrap Sharpe CIs (2,000 draws, 21-day blocks):
  timed net    +0.312  [-0.050, +0.656]  share<0 4.7%   <- includes zero
  timed gross  +0.566  [+0.204, +0.910]
  always-on    -0.268  [-0.645, +0.093]  share<0 93.0%


The gross number clears the bar (*t* = +3.04); the net one does not (*t* = +1.69, CI includes zero). The entire result lives in the gap between gross and net — the definition of a friction-sized effect, and the reason the next cell matters more than this one.

## The stale-quote decomposition — the check that decides it

Three of the seven pairs are thin OTC ADRs in New York (NPSNY, PROSY, SFTBY). Their closing prints can be stale by hours or simply unrefreshed. In a long/short pair a stale leg **manufactures** apparent mean reversion: today's spread is measured against a price that has not moved yet, and tomorrow it "reverts" when the quote catches up. That is a microstructure artefact with the same signature as the effect under test, so it has to be separated out.

> 💡 **In plain words** — if one of the two prices is yesterday's, the gap you measure is partly fictional, and it will look like it closes tomorrow.

In [4]:
rows = [('full panel (7)', R['timed_gross'], R['timed_gross_t'], R['timed_sharpe'], R['timed_t']),
        ('primary listings only (4)', R['clean_gross'], R['clean_gross_t'], R['clean_net'], R['clean_net_t']),
        ('the 3 OTC ADR pairs alone', R['adr_gross'], R['adr_gross_t'], R['adr_net'], R['adr_net_t'])]
print('%-28s %8s %7s %8s %7s' % ('sub-panel', 'gross', '(t)', 'net', '(t)'))
for tag, gs, gt, ns, nt in rows:
    print('%-28s %+8.3f %+7.2f %+8.3f %+7.2f' % (tag, gs, gt, ns, nt))
print()
print('pooled mean-reversion slope on the clean four: %+.4f (t %+.2f) -- still the wrong sign'
      % (R['clean_mr_beta'], R['clean_mr_t']))
print()
print('per-name standalone (nobody clears |t| = 2):')
print('  Prosus            %+.3f (t %+.2f)   <- a thin ADR' % (R['pn_prosus'], R['pn_prosus_t']))
print('  SoftBank          %+.3f (t %+.2f)   <- a thin ADR' % (R['pn_softbank'], R['pn_softbank_t']))
print('  Heineken Holding  %+.3f' % R['pn_heio'])
print('  Christian Dior    %+.3f' % R['pn_dior'])
print('  Liberty Broadband %+.3f' % R['pn_lbrd'])
print('  Bollore           %+.3f' % R['pn_bol'])

sub-panel                       gross     (t)      net     (t)
full panel (7)                 +0.566   +3.04   +0.312   +1.69
primary listings only (4)      +0.368   +1.87   +0.148   +0.76
the 3 OTC ADR pairs alone      +0.458   +2.16   +0.389   +1.85

pooled mean-reversion slope on the clean four: +0.0033 (t +1.27) -- still the wrong sign

per-name standalone (nobody clears |t| = 2):
  Prosus            +0.661 (t +1.61)   <- a thin ADR
  SoftBank          +0.398 (t +1.17)   <- a thin ADR
  Heineken Holding  +0.145
  Christian Dior    +0.087
  Liberty Broadband +0.066
  Bollore           +0.093


Gross *t* falls **+3.04 → +1.87** and net *t* to **+0.76** once the stale-quote names are removed; the two strongest standalone contributors are precisely the two thinnest ADRs. The pooled regression on the clean four is still the wrong sign (+0.0033, *t* = +1.27).

## The free parameters, swept whole

`enter` and `exit` in trailing-z units are the study's only genuinely free parameters — `k` and the hedge ratio are derived, costs and borrow are priced inputs — so an unswept default here would be a hidden choice. The 504-day standardisation window is a convention, and gets the same treatment. Both grids are reported whole rather than at the cell that was published.

In [5]:
print('enter/exit grid (%d cells with a hysteresis band):' % R['thr_cells'])
print('  gross t ranges [%+.2f, %+.2f]' % (R['thr_gross_lo'], R['thr_gross_hi']))
print('  net   t ranges [%+.2f, %+.2f]' % (R['thr_net_lo'], R['thr_net_hi']))
print('  cells with net t >= 2 : %d of %d   <- THE POINT'
      % (R['thr_cells_net2'], R['thr_cells']))
print('  headline cell (1.0/0.0) gross %+.3f (t %+.2f); best cell %+.3f (t %+.2f)'
      % (R['timed_gross'], R['timed_gross_t'], R['thr_best_gross'], R['thr_best_gross_t']))
print()
print('trailing-standardisation window (headline 504d):')
for w, t in [(252, R['zw252_t']), (378, R['zw378_t']), (504, R['timed_gross_t']),
             (756, R['zw756_t']), (1008, R['zw1008_t'])]:
    flag = '   <- headline, and the MAXIMUM of the sweep' if w == 504 else ''
    print('  %4dd  gross t %+.2f%s' % (w, t, flag))

enter/exit grid (17 cells with a hysteresis band):
  gross t ranges [+1.56, +3.43]
  net   t ranges [+0.08, +1.90]
  cells with net t >= 2 : 0 of 17   <- THE POINT
  headline cell (1.0/0.0) gross +0.566 (t +3.04); best cell +0.653 (t +3.43)

trailing-standardisation window (headline 504d):
   252d  gross t +2.42
   378d  gross t +2.46
   504d  gross t +3.04   <- headline, and the MAXIMUM of the sweep
   756d  gross t +1.89
  1008d  gross t +0.94


Two readings, and the study owes the reader both. In the threshold grid the published cell is *not* the best one (+0.653 at a lazier exit beats it), so the headline is not the top of a mined surface — but **no cell in the grid reaches a net *t* of 2**, so there is no threshold pair at which this is significant after friction. The window sweep is the less flattering of the two: 504 days is the **maximum**, and at 1008 days the gross *t* falls to +0.94 (Sharpe +0.160). Conclusion (a) is flat at every window, so nothing about the mean-reversion answer turns on it; the gross trading number — already the weaker half of the case — is worth about a point of *t* less anywhere else.

## Sweeps and robustness

In [6]:
print('cost sweep (one-way bps per leg, borrow fixed at 100 bps):')
for c, sh, t in [(0, R['cost0'], R['cost0_t']), (5, R['cost5'], R['cost5_t']),
                 (10, R['timed_sharpe'], R['timed_t']),
                 (25, R['cost25'], R['cost25_t']), (50, R['cost50'], R['cost50_t'])]:
    print('  %2d bps  Sharpe %+.3f (t %+.2f)' % (c, sh, t))
print()
print('borrow sweep (annualised bps on the short leg, cost fixed at 10 bps):')
for b, sh, t in [(0, R['borrow0'], R['borrow0_t']),
                 (100, R['timed_sharpe'], R['timed_t']),
                 (600, R['borrow600'], R['borrow600_t'])]:
    print('  %3d bps  Sharpe %+.3f (t %+.2f)' % (b, sh, t))
print()
print('era cut (2016): early %+.3f (t %+.2f)   late %+.3f (t %+.2f)'
      % (R['era_e_sh'], R['era_e_t'], R['era_l_sh'], R['era_l_t']))
print('leave-one-out : timed Sharpe stays in [%+.3f, %+.3f], t in [%+.2f, %+.2f]'
      % (R['loo_lo'], R['loo_hi'], R['loo_t_lo'], R['loo_t_hi']))
print('NAV-calibration sweep (k x0.8..1.2, other x0.5..1.5): Sharpe in [%+.3f, %+.3f]'
      % (R['calib_lo'], R['calib_hi']))
print('look-through hedge variant: net %+.3f (t %+.2f), gross %+.3f'
      % (R['lt_net'], R['lt_net_t'], R['lt_gross']))
print()
print('long-only, excess-of-cash: timed %+.3f | hold all holdcos %+.3f | hold the stakes %+.3f'
      % (R['lo_timed'], R['lo_holdcos'], R['lo_stakes']))
print('  HAC t on (timed - hold all holdcos): %+.2f  -> unhedged timing adds nothing'
      % R['lo_t_diff'])
print('cash-leg cross-check: ^IRX proxy %.2f%%/yr vs BIL total return %.2f%%/yr'
      % (R['irx_cagr'], R['bil_cagr']))

cost sweep (one-way bps per leg, borrow fixed at 100 bps):
   0 bps  Sharpe +0.510 (t +2.75)
   5 bps  Sharpe +0.411 (t +2.22)
  10 bps  Sharpe +0.312 (t +1.69)
  25 bps  Sharpe +0.016 (t +0.08)
  50 bps  Sharpe -0.467 (t -2.52)

borrow sweep (annualised bps on the short leg, cost fixed at 10 bps):
    0 bps  Sharpe +0.367 (t +1.99)
  100 bps  Sharpe +0.312 (t +1.69)
  600 bps  Sharpe +0.036 (t +0.20)

era cut (2016): early +0.319 (t +1.33)   late +0.256 (t +0.86)
leave-one-out : timed Sharpe stays in [+0.259, +0.346], t in [+1.41, +1.87]
NAV-calibration sweep (k x0.8..1.2, other x0.5..1.5): Sharpe in [+0.275, +0.321]
look-through hedge variant: net -0.040 (t -0.24), gross +0.174

long-only, excess-of-cash: timed +0.536 | hold all holdcos +0.655 | hold the stakes +0.636
  HAC t on (timed - hold all holdcos): +0.02  -> unhedged timing adds nothing
cash-leg cross-check: ^IRX proxy 1.45%/yr vs BIL total return 1.36%/yr


The NAV-calibration sweep clears the study's own biggest liability: scaling `k` from 0.8× to 1.2× and the `other` term from 0.5× to 1.5× moves the timed Sharpe only between +0.275 and +0.321, and the pooled mean-reversion *t* stays inside ±1.2 throughout. The trailing-z design did its job — the flat answer is not an artefact of the assumption table.

The look-through hedge variant turns negative (-0.040), but that is mechanical rather than informative: over-hedging a proportional gap leaves the pair net short the underlying through a 22-year bull market, so that number is about beta. It is reported because it shows how construction-sensitive this corner is.

## Live synthetic control — the machinery is unbiased

A panel of synthetic (stake, holdco) pairs where the discount lives in **logit space**, so it stays inside (0,1) without a reflecting barrier, plus an Itô correction that makes the discount itself a **martingale** when the mean-reversion pull is switched off. Both details matter: a barrier, or an uncorrected logistic drift, would have smuggled reversion into the very control that is supposed to have none. Planted OU must fire; the martingale null must not.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from holdco_nav import data, strategy as st
import numpy as np
for ss, tag in [(1.0, 'planted OU     '), (0.0, 'martingale null')]:
    rows = [st.synthetic_detect(data.synthetic_panel(signal_strength=ss, seed=957 + 37 * s)[0])
            for s in range(3)]
    t = np.array([r['pooled_t'] for r in rows])
    g = np.array([r['sharpe_timed_gross'] for r in rows])
    print('%s: pooled beta t mean %+6.2f   gross timed Sharpe mean %+.3f'
          % (tag, t.mean(), g.mean()))

planted OU     : pooled beta t mean -10.94   gross timed Sharpe mean +1.014


martingale null: pooled beta t mean  +0.37   gross timed Sharpe mean +0.052


On the frozen four-seed run in `docs/results.md` the planted world gives a pooled *t* of **-11.76** and a gross Sharpe of **+0.999**; the null gives **-0.38** and **+0.129**. The detector finds reversion when it is planted and is quiet when it is not, so the flat real-tape answer is a property of holdcos and not of the harness.

## Verdict

- **Signal — Weak.** The load-bearing half of the thesis is absent: pooled Driscoll-Kraay slope **-0.0002** (*t* = **-0.09**, R² = 0.0000), the right sign in **5/7** names — a coin flip — and two names where a wide gap got wider. The trading half posts a gross Sharpe of **+0.566** (*t* = **+3.04**) — but it falls to *t* = +1.69 at 10 bps a leg, with a bootstrap CI of [-0.050, +0.656] that includes zero, and it falls to gross *t* = **+1.87** / net **+0.76** once the three thin OTC ADR pairs are removed. Sweep the two thresholds that define the rule and **0 of 17** cells reach a net *t* of 2; sweep the standardisation window and the headline turns out to be the best of five. No single name clears |*t*| = 2 standalone; neither era does. What survives is a friction-sized reversal concentrated in the least liquid corner of the panel.
- **Tradability — Mirage.** Flat at 25 bps one-way (+0.016), -0.467 at 50; +0.036 at a 6% borrow; a net CI spanning zero; sign-flipping under the alternative hedge; and a required short leg in exactly the illiquid ADRs producing the signal. The patient version — own the discount and wait — returned **-1.68%/yr** over 22 years.
- **Survivorship, named.** The panel contains only holdcos that still exist and whose stake is still listed. Liberty TripAdvisor and Cannae, both classic wide-discount names, were dropped because their 2025 take-privates left no usable history — and a discount that ends in a buyout is a discount that *closed*. The bias therefore runs **in favour** of the thesis, and the thesis still failed.